# Pooled CMMD, FD-DINOv2, and Vendi using official implementations

This notebook evaluates **all images together**, recursively pooling images from every class directory.

| Metric | Implementation used | Direction |
|---|---|---|
| CMMD | Google Research's official `cmmd.main` script | Lower is better |
| FD-DINOv2 | Layer 6 AI's official DGM-Eval CLI with DINOv2 ViT-L/14 | Lower is better |
| Vendi-DINOv2 | Official `vendi_score` package applied to DGM-Eval's saved DINOv2 features | Higher is more diverse |

No metric formula is reimplemented in this notebook. The local code only discovers images, creates temporary flat views, runs the official tools, parses their results, and writes timestamped JSON.


## Official sources

- [Google Research CMMD](https://github.com/google-research/google-research/tree/master/cmmd)
- [Layer 6 AI DGM-Eval](https://github.com/layer6ai-labs/dgm-eval)
- [Official Vendi Score package](https://github.com/vertaix/Vendi-Score)

Google's CMMD is an official script, not a PyPI package. DGM-Eval is also most reliably run from its repository. Configure those checkout paths below.


## 1. One-time installation

Use an environment compatible with the official repositories. DGM-Eval recommends Python 3.10 and pins several older dependencies.

```bash
git clone https://github.com/google-research/google-research.git
git clone https://github.com/google-research/scenic.git
pip install -r /path/to/google-research/cmmd/requirements.txt

git clone https://github.com/layer6ai-labs/dgm-eval.git
pip install -e /path/to/dgm-eval

pip install vendi_score
```

CMMD downloads OpenAI CLIP ViT-L/14@336 weights through Scenic. DGM-Eval downloads DINOv2 ViT-L/14 weights on first use.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
import re
import subprocess
import sys
import tempfile
import time
from datetime import datetime
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

print("Python:", sys.version.split()[0])
print("vendi_score installed:", importlib.util.find_spec("vendi_score") is not None)


## 2. Configuration

Set the image roots and official repository checkout paths. Both image roots may contain nested `class_XXXX` directories; all discovered images are pooled.


In [ ]:
REAL_DIR = Path("/path/to/real")
SAMPLES_DIR = Path("/path/to/samples")

GOOGLE_RESEARCH_DIR = Path("/path/to/google-research")
SCENIC_DIR = Path("/path/to/scenic")
DGM_EVAL_DIR = Path("/path/to/dgm-eval")

PYTHON_EXECUTABLE = Path(sys.executable)
DEVICE = "cuda"  # cuda, cuda:0, cpu, or mps where supported by the tool.
BATCH_SIZE = 32
CMMD_BATCH_SIZE = None  # None selects the largest common divisor <= BATCH_SIZE.
NUM_WORKERS = 4
MAX_IMAGES = None  # None means every recursively discovered image.

OUTPUT_PATH = Path("pooled_official_metrics.json")
RUN_METRICS = False  # Set True after the preflight cell passes.


## 3. Discover images and verify tools

The official CMMD script reads only images directly inside each input directory. This notebook therefore creates temporary symlink directories containing unique filenames for every recursively discovered image. DGM-Eval receives the same staged directories.


In [ ]:
# Google Research CMMD officially supports PNG and JPEG inputs.
IMAGE_EXTENSIONS = {".jpeg", ".jpg", ".png"}


def discover_images(root: Path, max_images: int | None = None) -> list[Path]:
    if not root.is_dir():
        return []
    paths = sorted(
        path.resolve()
        for path in root.rglob("*")
        if path.is_file() and path.suffix.casefold() in IMAGE_EXTENSIONS
    )
    if max_images is not None:
        if max_images < 2:
            raise ValueError("MAX_IMAGES must be at least 2 or None.")
        paths = paths[:max_images]
    return paths


real_paths = discover_images(REAL_DIR.expanduser(), MAX_IMAGES)
sample_paths = discover_images(SAMPLES_DIR.expanduser(), MAX_IMAGES)


def largest_common_batch_size(first_count: int, second_count: int, limit: int) -> int:
    for candidate in range(min(first_count, second_count, limit), 0, -1):
        if first_count % candidate == 0 and second_count % candidate == 0:
            return candidate
    return 1


cmmd_batch_size = (
    largest_common_batch_size(len(real_paths), len(sample_paths), BATCH_SIZE)
    if CMMD_BATCH_SIZE is None and real_paths and sample_paths
    else CMMD_BATCH_SIZE
)
cmmd_uses_all_images = bool(
    cmmd_batch_size
    and len(real_paths) % cmmd_batch_size == 0
    and len(sample_paths) % cmmd_batch_size == 0
)

checks = {
    "real_images": len(real_paths),
    "sample_images": len(sample_paths),
    "google_cmmd": (GOOGLE_RESEARCH_DIR / "cmmd" / "main.py").is_file(),
    "scenic": (SCENIC_DIR / "scenic").is_dir(),
    "dgm_eval": (DGM_EVAL_DIR / "dgm_eval" / "__main__.py").is_file(),
    "vendi_score": importlib.util.find_spec("vendi_score") is not None,
    "cmmd_batch_size": cmmd_batch_size,
    "cmmd_uses_all_images": cmmd_uses_all_images,
}
display(checks)

paths_ready = len(real_paths) >= 2 and len(sample_paths) >= 2
tools_ready = all(
    checks[key]
    for key in ("google_cmmd", "scenic", "dgm_eval", "vendi_score")
)
tools_ready = tools_ready and cmmd_uses_all_images

if not paths_ready:
    print("Set REAL_DIR and SAMPLES_DIR to roots containing at least two images each.")
if not tools_ready:
    print("Complete the official tool installation and repository paths before running metrics.")


## 4. Official-tool orchestration helpers

These helpers do not calculate metrics. They stage image paths, invoke official commands, parse reported scores, and locate the DINOv2 feature files saved by DGM-Eval.


In [ ]:
def stage_images(paths: list[Path], output_dir: Path) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    for index, source in enumerate(paths):
        destination = output_dir / f"{index:08d}{source.suffix.casefold()}"
        try:
            destination.symlink_to(source)
        except OSError:
            os.link(source, destination)


def run_command(command: list[str], cwd: Path, env: dict[str, str] | None = None) -> str:
    print("Running:", " ".join(command))
    completed = subprocess.run(
        command,
        cwd=cwd,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(
            f"Official command failed with exit code {completed.returncode}: {' '.join(command)}"
        )
    return completed.stdout


def parse_single_float(pattern: str, text: str, metric_name: str) -> float:
    match = re.search(pattern, text, flags=re.IGNORECASE | re.MULTILINE)
    if match is None:
        raise ValueError(f"Could not parse {metric_name} from official command output")
    return float(match.group(1))


def load_dgm_representations(output_dir: Path, dataset_name: str) -> np.ndarray:
    matches = sorted(output_dir.glob(f"reps_{dataset_name}_dinov2_nimage-*_test.npz"))
    if len(matches) != 1:
        raise ValueError(
            f"Expected one DGM-Eval representation file for {dataset_name}, found: {matches}"
        )
    with np.load(matches[0], allow_pickle=True) as archive:
        return np.asarray(archive["reps"])


## 5. Run official CMMD, FD-DINOv2, and Vendi

Set `RUN_METRICS = True` after preflight passes. DGM-Eval extracts DINOv2 features once, uses them for official FD, and saves them. The official Vendi package then evaluates those exact generated and real feature arrays.


In [ ]:
metrics = None
timings = {}
official_outputs = {}

if not RUN_METRICS:
    print("Metrics skipped. Set RUN_METRICS = True after preflight passes.")
elif not paths_ready or not tools_ready:
    raise ValueError("Image paths and official tool installations must pass preflight.")
else:
    from vendi_score import vendi

    total_start = time.perf_counter()
    with tempfile.TemporaryDirectory(prefix="official-image-metrics-") as temporary_dir:
        temporary_root = Path(temporary_dir)
        staged_real = temporary_root / "real_pooled"
        staged_samples = temporary_root / "samples_pooled"
        dgm_output = temporary_root / "dgm_output"
        stage_images(real_paths, staged_real)
        stage_images(sample_paths, staged_samples)

        cmmd_env = os.environ.copy()
        existing_pythonpath = cmmd_env.get("PYTHONPATH", "")
        cmmd_env["PYTHONPATH"] = os.pathsep.join(
            value
            for value in (str(GOOGLE_RESEARCH_DIR.resolve()), str(SCENIC_DIR.resolve()), existing_pythonpath)
            if value
        )
        stage_start = time.perf_counter()
        cmmd_output = run_command(
            [
                str(PYTHON_EXECUTABLE),
                "-m",
                "cmmd.main",
                str(staged_real),
                str(staged_samples),
                f"--batch_size={cmmd_batch_size}",
                f"--max_count={max(len(real_paths), len(sample_paths))}",
            ],
            cwd=GOOGLE_RESEARCH_DIR.resolve(),
            env=cmmd_env,
        )
        timings["official_cmmd"] = time.perf_counter() - stage_start
        cmmd = parse_single_float(
            r"The CMMD value is:\s*([-+0-9.eE]+)",
            cmmd_output,
            "CMMD",
        )
        official_outputs["cmmd"] = cmmd_output

        stage_start = time.perf_counter()
        dgm_output_text = run_command(
            [
                str(PYTHON_EXECUTABLE),
                "-m",
                "dgm_eval",
                str(staged_real),
                str(staged_samples),
                "--model",
                "dinov2",
                "--arch",
                "vitl14",
                "--metrics",
                "fd",
                "--device",
                DEVICE,
                "--batch_size",
                str(BATCH_SIZE),
                "--num-workers",
                str(NUM_WORKERS),
                "--nsample",
                str(max(len(real_paths), len(sample_paths))),
                "--output_dir",
                str(dgm_output),
                "--save",
                "--no-load",
            ],
            cwd=DGM_EVAL_DIR.resolve(),
        )
        timings["official_fd_dinov2_and_features"] = time.perf_counter() - stage_start
        fd_dinov2 = parse_single_float(r"^fd:\s*([-+0-9.eE]+)", dgm_output_text, "FD-DINOv2")
        official_outputs["dgm_eval"] = dgm_output_text

        real_dino = load_dgm_representations(dgm_output, staged_real.name)
        sample_dino = load_dgm_representations(dgm_output, staged_samples.name)

        stage_start = time.perf_counter()
        vendi_real = float(vendi.score_dual(real_dino, normalize=True))
        vendi_generated = float(vendi.score_dual(sample_dino, normalize=True))
        timings["official_vendi_score"] = time.perf_counter() - stage_start

    timings["total"] = time.perf_counter() - total_start
    metrics = {
        "cmmd": cmmd,
        "fd_dinov2": fd_dinov2,
        "vendi_generated": vendi_generated,
        "vendi_real": vendi_real,
        "vendi_generated_to_real_ratio": (
            vendi_generated / vendi_real if vendi_real > 0 else None
        ),
    }
    print("Official metric calculation complete.")


## 6. Display and save timestamped results

The JSON includes metric values, repository paths, image counts, timing, and the raw official command output for auditability.


In [ ]:
if metrics is None:
    print("No results to display. Run Step 5 with RUN_METRICS = True first.")
else:
    rows = [
        "| Metric | Value | Direction |",
        "|---|---:|---|",
        f"| Official CMMD | {metrics['cmmd']:.6f} | Lower is better |",
        f"| Official FD-DINOv2 | {metrics['fd_dinov2']:.6f} | Lower is better |",
        f"| Official Vendi-DINOv2 generated | {metrics['vendi_generated']:.6f} | Higher is more diverse |",
        f"| Official Vendi-DINOv2 real | {metrics['vendi_real']:.6f} | Reference diversity |",
        f"| Vendi generated / real | {metrics['vendi_generated_to_real_ratio']:.6f} | Context only |",
    ]
    display(Markdown("\n".join(rows)))

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_output_path = OUTPUT_PATH.expanduser().resolve()
    output_path = base_output_path.with_name(
        f"{base_output_path.stem}_{timestamp}{base_output_path.suffix}"
    )
    payload = {
        "created_at_local": datetime.now().astimezone().isoformat(),
        "real_dir": str(REAL_DIR.expanduser().resolve()),
        "samples_dir": str(SAMPLES_DIR.expanduser().resolve()),
        "real_images": len(real_paths),
        "sample_images": len(sample_paths),
        "used_all_images": MAX_IMAGES is None,
        "max_images": MAX_IMAGES,
        "device": DEVICE,
        "batch_size": BATCH_SIZE,
        "cmmd_batch_size": cmmd_batch_size,
        "official_sources": {
            "cmmd": "https://github.com/google-research/google-research/tree/master/cmmd",
            "fd_dinov2": "https://github.com/layer6ai-labs/dgm-eval",
            "vendi": "https://github.com/vertaix/Vendi-Score",
        },
        "official_checkouts": {
            "google_research": str(GOOGLE_RESEARCH_DIR.expanduser().resolve()),
            "scenic": str(SCENIC_DIR.expanduser().resolve()),
            "dgm_eval": str(DGM_EVAL_DIR.expanduser().resolve()),
        },
        "metrics": metrics,
        "timing_seconds": timings,
        "official_command_output": official_outputs,
    }

    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = output_path.with_suffix(output_path.suffix + ".tmp")
    temporary_path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")
    temporary_path.replace(output_path)
    print(f"Saved: {output_path}")
    print(f"Total runtime: {timings['total'] / 60:.2f} minutes")


## Notes

- Google CMMD drops incomplete batches internally. The notebook automatically chooses a common batch divisor unless `CMMD_BATCH_SIZE` is set explicitly, ensuring every selected image is used.
- On multi-device JAX systems, the CMMD batch size must also be divisible by the visible JAX device count. Restrict visible devices or set an appropriate common divisor if the official script reports this error.
- DGM-Eval may randomly subsample when `MAX_IMAGES` or its `nsample` is lower than the available count. This notebook stages exactly the selected images so no additional reduction is intended.
- Vendi is computed by the official package on the exact DINOv2 representations saved by DGM-Eval, avoiding a second feature extractor or preprocessing mismatch.
- Compare runs only when the official tool revisions, model architecture, preprocessing, reference set, and sample count are identical.
